# MongoDB Atlas Connector Sample

Load a MongoDB Atlas collection into a Delta table: a full load on the first run, then incremental loads on a Date watermark field. Read-only against MongoDB. Replace all `<PLACEHOLDER>` values before running.

## Prerequisites

1. Upload `mongodb_client.py` (the only helper file, from this folder) to a workspace folder and set `HELPER_DIR` below to that folder.
2. Provide `MONGODB_URI` (`mongodb+srv://<user>:<password>@<cluster>.mongodb.net/`, for a read-only database user) as an environment variable on the cluster, or as an OCI Vault secret with `OCI_VAULT_ID` set (this also needs the `oci` package, and `OCI_COMPARTMENT_ID` if the secret is not in the tenancy root). Do not type the URI into a cell: it contains the password.
3. Atlas only accepts connections from IPs on the project's IP access list. Add the cluster's outbound IP there first; changes take a minute or two to apply.
4. The cluster needs network access to Maven Central (`repo1.maven.org`) to download the MongoDB Spark Connector jars. No PyPI package is needed.

## Configuration

In [ ]:
import sys

HELPER_DIR = "<WORKSPACE_PATH_TO_HELPER_FOLDER>"  # contains mongodb_client.py
TARGET = "<CATALOG>.<SCHEMA>.<TABLE>"
DATABASE = "<DATABASE>"
COLLECTION = "<COLLECTION>"
WATERMARK_FIELD = "<DATE_FIELD>"  # must hold BSON Dates (not strings); None = full re-read every run
STRING_FIELDS = []                # fields whose type differs between documents, read as text
SAMPLE_SIZE = 10000               # first run only: documents sampled to infer the schema
OVERLAP_SECONDS = 300
sys.path.insert(0, HELPER_DIR)

## Load the Connector and Check the Connection

The SHA-256-pinned connector and driver jars are downloaded from Maven Central to `/tmp` and loaded into the running session, so no restart is needed. The connection check reads one sampled document: it proves network access, the IP access list, auth and read access, and fails with a hint instead of a bare Py4J error.

In [ ]:
import mongodb_client as m

m.load_mongo_connector(spark)  # safe to re-run

uri = m.credentials_from_env()
m.check_connection(spark, uri, DATABASE, COLLECTION)
print("connection ok")

## Read Documents

Later runs reuse the target table's schema, so a type the sample happens to guess differently cannot change the table under the `MERGE`. The watermark is read as epoch microseconds inside the helper: a collected TIMESTAMP comes back in the Python process's local timezone, which would shift it.

In [ ]:
from datetime import datetime, timezone

since, schema = None, None
if spark.catalog.tableExists(TARGET):
    schema = spark.table(TARGET).schema
    if WATERMARK_FIELD:
        since = m.latest_watermark(spark, TARGET, WATERMARK_FIELD)

until = datetime.now(timezone.utc) if WATERMARK_FIELD else None  # fixed for this run
df = m.read_collection(
    spark, uri, DATABASE, COLLECTION,
    schema=schema, string_fields=STRING_FIELDS, sample_size=SAMPLE_SIZE,
    watermark_field=WATERMARK_FIELD, since=since, until=until, overlap_seconds=OVERLAP_SECONDS,
)

## Write to Delta

The overlap window re-reads a few documents on purpose, so later runs `MERGE` on `_id` to remove duplicates. The actual MongoDB read happens here, so errors are explained here too.

In [ ]:
try:
    if not spark.catalog.tableExists(TARGET):
        df.write.format("delta").saveAsTable(TARGET)
    else:
        df.createOrReplaceTempView("incoming_document")
        spark.sql(
            f"MERGE INTO {TARGET} t USING incoming_document s ON t._id = s._id "
            "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *"
        )
except Exception as exc:
    raise m.explain_error(exc, uri) from None
print("documents in target:", spark.table(TARGET).count())

## Limits

- A watermark does not see deletes in MongoDB.
- `WATERMARK_FIELD` must hold BSON Dates. A string field never matches a Date bound, so an incremental read of it returns nothing.
- The schema is inferred once, on the first run, then widened (decimals to `decimal(38, >=10)`, ints to longs), because the connector's sampling otherwise silently nulls values wider than the sample. A field whose type varies between documents fails the read with `UnknownReason`: add it to `STRING_FIELDS`, drop the target table and rerun.
- Later runs reuse the target table's schema, so fields that first appear after the first run are not added. Drop the table and rerun to pick them up.
- To check a column for lost values, count on the target table: `df.filter(col.isNull())` is pushed down to MongoDB and counts only server-side nulls.
- A free Atlas M0 cluster is for learning and testing, not production volume.